# SaleOps: SQL Analysis of Car Sales Data 
## Notebook Declaration 
This notebook contains SQL‑based analysis of car sales data.  
It demonstrates the use of advanced SQL techniques — including CTEs, window functions, and aggregations — to explore:</br>
1. Revenue contribution by product line and warehouse</br>
2. Pricing differences across warehouses</br>
3. Client type purchasing behavior (Retail vs. Wholesale)</br>
4. Product line prioritization for revenue growth</br>
5. Operational performance insights by warehouse</br>
The analysis focuses on revenue patterns and customer behavior, as payment fees are negligible and no cost data is included. The goal is to showcase SQL query writing ability and derive actionable business insights from structured sales data.

In [2]:
import pandas as pd 
import prettytable 
import sqlite3 
prettytable.DEFAULT = 'DEFAULT' 
%load_ext sql 
%sql sqlite:///sales.db
df = pd.read_excel('Sale.xlsx') 
con = sqlite3.connect('sales.db') 
cur = con.cursor() 

df.to_sql('sales',con,  if_exists='replace', index= False , method ='multi') 

1000

## DataSet Overview 

In [4]:
df.head() 

,Index,Order_Number,Month,Date,Warehouse,Client_Type,Product_Line,Quantity,Unit_Price,Total,Payment,Payment_Fee
0,0,N1,June,2021-06-01,North,Retail,Braking system,9,19.29,173.61,Cash,0.00
1,1,N2,June,2021-06-01,North,Retail,Suspension & traction,8,32.93,263.45,Credit card,0.03
2,2,N3,June,2021-06-01,North,Wholesale,Frame & body,16,37.84,605.44,Transfer,0.01
3,3,N4,June,2021-06-01,North,Wholesale,Suspension & traction,40,37.37,1494.80,Transfer,0.01
4,4,N5,June,2021-06-01,North,Retail,Frame & body,6,45.44,272.61,Credit card,0.03


<h2> Analysis Questions </h2>

The following questions analyze sales performance across warehouses, product lines and client types 
They are organized to progress from basic aggregations to deeper, more interpretive insights, creating a logical flow of analysis.


<h2>Table of Contents</h2>
<div class="alert alert-block alert-info" style="margin-top: 20px">
<ol>
    <li><a href="#1.What-is-the-driving-Dominant-Product_Line-and-its-Contribution-on-a-warehouse?">
        What is the driving Dominant Product_Line and its Contribution on a warehouse?
    </a>
        <ol>
            <li><a href="#1-i-product-lines-maximize-revenue">
                (i) Which product lines should the company prioritize to maximize revenue?
            </a></li>
            <li><a href="#1-ii-revenue-differences-warehouses">
                (ii) What factors drive revenue differences between warehouses?
            </a></li>
        </ol>
    </li>
    <li><a href="#2.Which-warehouse-has-higher-quantity-sold-but-lower-revenue,-indicating-lower-pricing?">
        Which warehouse has higher quantity sold but lower revenue, indicating lower pricing?
    </a></li>
    <li><a href="#3.How-does-client-type-influence-purchasing-behavior-and-profitability?">
        How does client type influence purchasing behavior and profitability?
    </a></li>
    <li><a href="#4.Which-client-type-generates-more-revenue-per-order?">
        Which client type generates more revenue per order?
    </a></li>
    <li><a href="#5.What-operational-improvements-can-be-made-based-on-warehouse-performance?">
        What operational improvements can be made based on warehouse performance?
    </a></li>
</ol>
</div>
<hr>

## 1. What is the driving Dominant Product_Line and its Contribution on a warehouse ? 

In [8]:
%%sql
WITH product AS ( 
    SELECT 
        product_line, 
        warehouse, 
        SUM(quantity) AS total_quantity, 
        ROUND(SUM(total) - SUM(quantity) * SUM(payment_fee), 1) AS net_revenue
    FROM sales 
    GROUP BY  product_line,warehouse
) 
SELECT 
    product_line, 
    warehouse, 
    total_quantity, 
    net_revenue,
    SUM(net_revenue) OVER(PARTITION BY warehouse) AS total_warehouse_revenue, 
    ROUND(
        net_revenue * 100.0 / SUM(net_revenue) OVER(PARTITION BY warehouse),
        1
    ) AS contribution
FROM product
ORDER BY contribution DESC 


 * sqlite:///sales.db
Done.


product_line,warehouse,total_quantity,net_revenue,total_warehouse_revenue,contribution
Suspension & traction,North,866,28391.6,95502.2,29.7
Frame & body,North,675,27581.1,95502.2,28.9
Suspension & traction,Central,980,30280.0,133059.8,22.8
Frame & body,West,246,10431.3,45721.8,22.8
Suspension & traction,West,299,10027.7,45721.8,21.9
Frame & body,Central,698,28777.2,133059.8,21.6
Braking system,West,501,9493.5,45721.8,20.8
Engine,Central,449,26917.4,133059.8,20.2
Electrical system,Central,838,20343.7,133059.8,15.3
Electrical system,North,607,14361.1,95502.2,15.0


Across all the warehouse ,  <b>Suspension & traction </b> leads as an dominant product_line in sales. 

<h3 id="1-i-product-lines-maximize-revenue">
(i) Which product lines should the company prioritize to maximize revenue?
</h3>

To get more profit the company should <b>shift focus on Suspension & Traction</b> for its strong revenue contribution while <b>expanding focus on Frame & Body</b>, which has the combination of high total revenue with stronger unit pricing. 

Frame and Body represents the best balance between demand and pricing power, showing it a <b>key product line</b> for profit growth. 

<h3 id="1-ii-revenue-differences-warehouses">
(ii) What factors drive revenue differences between warehouses?
</h3>

In [11]:
%%sql 
SELECT warehouse, 
    ROUND(SUM(total) - SUM(payment_fee),2) AS net_revenue,
    SUM(quantity) AS total_quantity,
    ROUND(AVG(unit_price),2)  as avg_unit_price 
FROM sales 
GROUP BY warehouse
ORDER BY net_revenue DESC;  

 * sqlite:///sales.db
Done.


Warehouse,net_revenue,total_quantity,avg_unit_price
Central,141972.13,4527,30.52
North,100196.26,3254,30.35
West,46922.59,1614,29.75


The primary revenue difference is highly influenced by the place showing the <b>Location specific demand </b>, explaining why the central outperfroms than the others. In addition, the price differences contribute only marginally to overall revenue disaprities indicating the demand and sales volume and the dominant factors. 

------------------------

## 2.Which warehouse has higher quantity sold but lower revenue, indicating lower pricing?

### Looking for the higher quantity but lower revenue
First, when looking for which warehouse sells more units but less profit overall, suggesting lower pricing, there is not much different. 
<br> <b>Comparing North vs West </b>: North sell more units (3.2k vs 1.6k) and also earns more revenue. 
<br><b>Comparing Central vs North</b> : Central sells more units and earns more revenue. 
<br><b>Comparing Central vs West</b>:  Central sells more units and earns more revenue. 
So , none of the warehouse show a pattern of higher qunatity sells but lower revenue.   

The avgerage prices are very close to each other <b>~30</b>, suggesting pricing is fairly consistent across warehouses, with only slight differences.The lower revenue in West in simply caused by the quantity it sold, not because of lower pricing, which indicate there is <b>no lower pricing. </b>

---------------

## 3.How does client type influence purchasing behavior and profitability?

In [18]:
%%sql 
SELECT 
    Client_Type, 
    warehouse, 
    ROUND ( SUM(total) - SUM(quantity) * SUM(Payment_Fee) ,2  ) AS net_revenue,
    SUM(quantity) AS total_quantity ,
    ROUND ( AVG(Unit_Price),2 ) AS avg_unit_price 
FROM sales 
GROUP BY Client_Type, warehouse 

 * sqlite:///sales.db
Done.


Client_Type,Warehouse,net_revenue,total_quantity,avg_unit_price
Retail,Central,43429.38,2039,30.28
Retail,North,32936.96,1394,30.24
Retail,West,21462.37,782,30.38
Wholesale,Central,76144.84,2488,31.31
Wholesale,North,56634.07,1860,30.72
Wholesale,West,22394.82,832,27.49


Wholesale generally drive to be more profitable in central and north, <b>due to its large purchase. </b>

<b>Retail Clients in West</b> contribute make more revenue per unit, even though Wholesale buys more units, which highlight a location-specific pricing or demand effect.

<h4>1.Purchasing Behavior</h4>  Bulk   </br> 
<h4>2.Market </h4>  Whether wholesale or retail, the profit depends on the warehouse's location showing the<b> "Local Dynamics" </b> Central and North benefiting from Wholesale, while West earn more from retial despite low overall quantity.
<h4>3.Strategy</h4>
If a company wants to buy in bulk, West warehouse should be considered, making best area to source. Reselling those goods in Central where demand and sales volumes are the highest, would maximize the profit.

------------

## 4.Which client type generates more revenue per order?

In [24]:
%%sql 
SELECT 
    Client_Type, 
    ROUND(AVG(net_revenue), 2 ) AS avg_revenue_per_order
FROM (
    SELECT 
        client_type, 
        order_number, 
        SUM(total)-SUM(payment_fee) AS net_revenue
    FROM sales 
    GROUP BY client_type, order_number 
)t 
GROUP BY client_type 
ORDER BY avg_revenue_per_order DESC ;

 * sqlite:///sales.db
Done.


client_type,avg_revenue_per_order
Wholesale,709.51
Retail,167.03


Wholesale Clients consistently generate higher revenue per order, which reflect <b>the bulk buying nature.</b>

This findings complements the finding in "Question3" , where wholesale buyers dominated profitablity in Central and North as of the volume it sold, while Retail client in the West benefited from higher unit pirces. 

Together, these results show that profitability factors are driven by the client type: showing the Wholesale profitability scales through order size, whereas retail profitability depends more on pricing and local demand conditions. 

--------

## 5.What operational improvements can be made based on warehouse performance?

In [28]:
%%sql 
WITH warehouse_summary AS ( 
    SELECT 
        warehouse , 
        SUM(quantity) AS total_quantity, 
        ROUND(SUM(total)-COUNT(order_number)*(payment_fee),2) AS real_value 
    FROM sales 
    GROUP BY warehouse 
    ) 
SELECT * ,
    ROUND ((real_value/ total_quantity) ,2 ) AS avg_revenue_per_unit 
FROM warehouse_summary 
ORDER BY real_value DESC;

 * sqlite:///sales.db
Done.


warehouse,total_quantity,real_value,avg_revenue_per_unit
Central,4527,141982.88,31.36
North,3254,100203.63,30.79
West,1614,46921.09,29.07


### Warehouse Performance Summary
<b>Central warehouse</b> demonstrates the <b>strongest operational performance</b> driven by both highest sale volume and revenue efficiency per unit. 

<b>North Warehouse</b> shows <b>competitive performance</b> but may benefit from pricing or product mix optimization. 

<b>West Warehouse underperforms</b> across all metrics, suggesting a need for operational improvements such as inventory optimization, cost reduction, or reallocation of high-performing product lines. 

<h3>Supporting Breakdown </h3> 
<img src="Breakdown.png" width="400">

<h1>END</h1>

### Author 
<a href="https://www.linkedin.com/in/pyae-khant-kyaw-591726390/" target="_blank">Pyae Khant Kyaw</a>
# <h3 align="center"> ©All rights reserved. <h3/>

---

### Additional Findings

### 1. Top 2 Product Line per warehouse 

In [36]:
%%sql
SELECT 
    Product_Line,
    Warehouse,
    net_revenue,
    total_quantity
FROM (
    SELECT 
        Product_Line, 
        Warehouse, 
        ROUND(SUM(Total) - SUM(quantity) * SUM(payment_fee), 2) AS net_revenue, 
        SUM(Quantity) AS total_quantity,
        DENSE_RANK() OVER (
            PARTITION BY Warehouse 
            ORDER BY ROUND(SUM(Total) - SUM(quantity) * SUM(payment_fee), 2) DESC
        ) AS rank
    FROM sales
    GROUP BY Product_Line, Warehouse
) t
WHERE rank <= 2 ;

 * sqlite:///sales.db
Done.


Product_Line,Warehouse,net_revenue,total_quantity
Suspension & traction,Central,30280.01,980
Frame & body,Central,28777.18,698
Suspension & traction,North,28391.63,866
Frame & body,North,27581.11,675
Frame & body,West,10431.33,246
Suspension & traction,West,10027.7,299


### 2. Finding the Weakest Product Line per Warehouse Ranking

In [38]:
%%sql
WITH weakest AS ( 
    SELECT 
        product_line, 
        warehouse, 
        ROUND(Sum(total) - sum(quantity) * sum (payment_fee) , 2 )  AS net_revenue
    FROM sales 
    GROUP BY Product_Line, Warehouse) ,
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY Warehouse 
            ORDER BY net_revenue ASC
        ) AS rank
    FROM weakest 
) 
SELECT * 
FROM ranked
WHERE rank = 1 


 * sqlite:///sales.db
Done.


product_line,warehouse,net_revenue,rank
Miscellaneous,Central,11580.66,1
Engine,North,7190.37,1
Engine,West,3432.07,1


# <h3 align="center"> ©All rights reserved. <h3/>

---